
# Automatic BB

### Image comparing

In [2]:
import cv2
import torch
import time
import numpy as np

img_path2 = "editing/samples/sk111.png"
img_path1 = "editing/samples/sk222.png"
gen_img1 = "editing/state/gen_img_1.png"
gen_img2 = "editing/state/gen_img_2.png"

# test_img = "editing/state/gen_img_1.png"

%load_ext autoreload
%autoreload 2

In [11]:
img1 = cv2.imread(img_path1, cv2.IMREAD_COLOR)
img2 = cv2.imread(img_path2, cv2.IMREAD_COLOR)

diff = cv2.absdiff(img1, img2)  # per-pixel absolute difference
cv2.imwrite("editing/samples/diff1.png", diff)

True

In [14]:
import cv2

# img_path1 = "editing/samples/image_1.png"
# img_path2 = "editing/samples/image_2.png"

img1 = cv2.imread(img_path1, cv2.IMREAD_COLOR)
img2 = cv2.imread(img_path2, cv2.IMREAD_COLOR)

# 1. Get the absolute per-pixel difference
diff = cv2.absdiff(img1, img2)

# 2. Convert the difference to grayscale
# This merges the BGR channels into a single intensity map so we can threshold it cleanly.
gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

# 3. Define your threshold value (0 to 255)
# A value of 30 is a great starting point to ignore tiny sensor noise or jpeg compression.
# Increase this number if it's picking up too much background noise.
threshold_value = 200 

# 4. Apply the Binary Threshold
# If the difference is > 30, force it to 255 (Pure White). 
# If it's <= 30, force it to 0 (Pure Black).
_, thresh_mask = cv2.threshold(gray_diff, threshold_value, 255, cv2.THRESH_BINARY)

# Save both so you can compare the raw difference vs the clean threshold mask
cv2.imwrite("editing/samples/img_diff1_raw.png", diff)
cv2.imwrite("editing/samples/diff1.png", thresh_mask)

True

In [14]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
from depth_anything_3.api import DepthAnything3
import cv2
import torch
import time
import numpy as np

class DA3:
    def __init__(self):
        # model = DepthAnything3.from_pretrained("depth-anything/DA3MONO-LARGE")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # model = DepthAnything3.from_pretrained("depth-anything/DA3NESTED-GIANT-LARGE-1.1")
        model = DepthAnything3.from_pretrained("depth-anything/DA3-BASE")
        model = model.to(device)
        model.eval()
        print(f"Model loaded on {device}")
        
        self.run_times = []
        self.model = model
        self.device = device

    def forward(self, img_path):
        start = time.perf_counter()

        original_image = cv2.imread(img_path)
        H_orig, W_orig = original_image.shape[:2]
        longest_edge = max(H_orig, W_orig)
        optimal_res = int(round(longest_edge / (14.0)) * 14)
        process_res = min(optimal_res, 1330)
        print(f"Running native inference at process_res: {process_res}")

        prediction = self.model.inference(image=[img_path], process_res=process_res)

        depth = prediction.depth[0] # Depth in [m].
        print("depth:", depth.shape)

        depth_resized = cv2.resize(
            depth, 
            (W_orig, H_orig), 
            interpolation=cv2.INTER_LINEAR  # Change to cv2.INTER_NEAREST if edges stretch in 3D
        )
        depth_resized = depth_resized.astype(np.float32)
        print("depth_resized:", depth_resized.shape)

        if prediction.intrinsics is None:
            return depth_resized, None, None, None, None, None, 

        H_pred, W_pred = depth.shape

        scale_x = W_orig / W_pred
        scale_y = H_orig / H_pred
        fx = prediction.intrinsics[0, 0, 0] * scale_x
        fy = prediction.intrinsics[0, 1, 1] * scale_y
        cx = prediction.intrinsics[0, 0, 2] * scale_x
        cy = prediction.intrinsics[0, 1, 2] * scale_y

        h, w = depth_resized.shape
        SENSOR_HEIGHT_MM = 24.0  # Standard full-frame sensor height
        focal_length = (fy / h) * SENSOR_HEIGHT_MM
        print("focal_length:", focal_length)

        end = time.perf_counter()
        runtime = round(end - start, 3)
        self.run_times.append(runtime)

        return depth_resized, focal_length, fx, fy, cx, cy

    def mean_runtime (self):
        arr = np.array(self.run_times)
        return arr.mean()

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


In [6]:
da3 = DA3()

[INFO ] using MLP layer as FFN
Model loaded on cuda


In [7]:
depth, focal_length, fx, fy, cx, cy = da3.forward(gen_img1)

Running native inference at process_res: 518
[INFO ] Processed Images Done taking 0.04952287673950195 seconds. Shape:  torch.Size([1, 3, 518, 518])
[INFO ] Model Forward Pass Done. Time: 0.5300993919372559 seconds
[INFO ] Conversion to Prediction Done. Time: 0.002002239227294922 seconds
depth: (518, 518)
depth_resized: (512, 512)
focal_length: 52.584610279922785


In [4]:
fx, fy, cx, cy = (1209.865588803089, 1195.9805743243244, 256.0, 256.0)

In [6]:
import os
from editing.bbox.render import render_single_view_for_bbox, get_cam_to_mesh_matrix
from editing.reconstruction import process_2d_changes

process_2d_changes(img_path1, img_path2, gen_img1, save_dir="editing/state")

mesh_path = "editing/state/mesh_1.glb"

# Validate file presence before running
if not os.path.exists(mesh_path):
    print(f"Error: Mesh file not found at: '{mesh_path}'")
    exit()

# fx, fy, cx, cy = (1209.865588803089, 1195.9805743243244, 256.0, 256.0)

# Highly-aligned Camera Angles (keep these to align render view to your photo perspective)
target_az = -np.pi / 1.0
target_el = -np.pi / 10.5       

img1, depth_map, camera_pose, scale, center = render_single_view_for_bbox(mesh_path, fx, fy, cx, cy, target_az, target_el)
# print(depth_map[200, 200])
# plt.imsave('editing/state/depth_image.png', depth_map, cmap='plasma')

cv2.imwrite(
    "editing/state/render_from_mesh.png",
    cv2.cvtColor(img1, cv2.COLOR_RGB2BGR)
)

M = get_cam_to_mesh_matrix(camera_pose, scale, center)

In [25]:
from editing.bbox.correspondence import ImageMatcher
from editing.bbox.warp import estimate_homography, warp_difference, save_debug

render = cv2.imread("editing/state/render_from_mesh.png")
flux = cv2.imread("editing/state/gen_img_1.png")
diff = cv2.imread("editing/state/changed_part.png")
render = cv2.cvtColor(
    render,
    cv2.COLOR_BGR2RGB
)
flux = cv2.cvtColor(
    flux,
    cv2.COLOR_BGR2RGB
)
diff = cv2.cvtColor(
    diff,
    cv2.COLOR_BGR2RGB
)
matcher = ImageMatcher()

pts_render, pts_flux, conf = matcher.match(
    render,
    flux
)

H, inliers = estimate_homography(
    pts_render,
    pts_flux,
    conf
)

aligned = warp_difference(
    diff,
    H,
    (
        render.shape[1],
        render.shape[0]
    )
)

save_debug(
    aligned,
    "editing/state/aligned_difference.png"
)

Using matches: 317
Homography inliers: 103 / 317


In [26]:
from editing.reconstruction import generate_pcd
import open3d as o3d
from PIL import Image
import numpy as np

rgb_image = Image.open("editing/state/aligned_difference.png")
rgb_array = np.array(rgb_image)
print(rgb_array.shape)

base_mask = np.any(rgb_array > 0, axis=-1)
# base_mask = np.load("editing/state/changed_part.npy")

changed_pcd = generate_pcd(rgb_array, depth_map, base_mask, fx, fy, cx, cy)
o3d.io.write_point_cloud(os.path.join("editing/state", "changed_part_pc.ply"), changed_pcd, write_ascii=False)

(512, 512, 3)


True

In [5]:
from editing.reconstruction import extract_changes

BB_Corners, changed_pcd = extract_changes(img_path1, img_path2, gen_img1, depth, fx, fy, cx, cy)
BB_Corners

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Pipeline complete. Files saved to 'editing/state'.


{'corners': [[-0.049754162092637555,
   -0.055014104099185224,
   -0.7419428129562537],
  [0.06320729036428412, 0.007951320589584988, -1.0266678887031606],
  [-0.08781943244852672, 0.07107436181011181, -0.7291609897634039],
  [0.04589393923636495, -0.030534154201040137, -0.6985819011708028],
  [0.12079012133739746, 0.15851973639702713, -0.9705251537248598],
  [0.007828668880475786, 0.0955543117082569, -0.685800077977953],
  [0.1588553916932866, 0.03243127048773008, -0.9833069769177096],
  [0.025142020008394952, 0.134039786498882, -1.0138860655103108]]}

In [6]:
from editing.test import run_alignment

M = run_alignment("kv_cache/mesh_pass1.glb", "editing/output/bg_free_full_pc.ply")
M

array([[ 0.14319186,  0.04926849, -0.15656088, -1.76497045],
       [-0.04163445,  0.21187595,  0.02859651,  0.26524606],
       [ 0.15876167,  0.01112668,  0.1487062 ,  1.57780581],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [33]:
from editing.reconstruction import transform_bounding_box

transformed_bb = transform_bounding_box(changed_pcd, M, padding=0.03)
# trellis_bb = convert_bb_to_trellis(transformed_bb)
transformed_bb

{'min': [-0.16177141225081734, -0.12800568429130918, -0.5189288139904973],
 'max': [0.16915169605199007, 0.22869788021839996, 0.29742939587725425]}

In [39]:
from editing.reconstruction import get_trellis_latent_mask, get_trellis_bb

print(get_trellis_latent_mask(transformed_bb))
print(get_trellis_bb("editing/state/mesh_1.glb", transformed_bb))

{'min': [0.3719943157086908, 0.20257060412274575, 0.33822858774918263], 'max': [0.7286978802184, 1.0, 0.6691516960519901]}
{'min': [0.18742645939669464, 0.20284093208149612, 0.0], 'max': [1.0, 1.0, 1.0]}


In [11]:
from editing.reconstruction import visualize_bounding_box

visualize_bounding_box("editing/output/bg_free_full_pc.ply", BB_Corners)

Visualizing: Oriented Bounding Box (Green)
Opening interactive viewer. Close the window to continue script execution.


In [35]:
visualize_bounding_box("editing/output/generated.ply", transformed_bb)

Visualizing: Axis-Aligned Bounding Box (Red)
Opening interactive viewer. Close the window to continue script execution.
